# Оценка личностных характеристик авторов текстов с помощью LLM

В данном проекте исследуется способность больших языковых моделей (LLM)
определять уровень интроверсии–экстраверсии автора текста на русском языке.

В качестве эталонных данных используется корпус **rusidiolect**,
в котором тексты размечены по шкале интроверсии–экстраверсии (0–100).

Эксперимент проводится в формате few-shot prompting с одинаковым промптом
для всех моделей. Для доступа к моделям используется платформа **OpenRouter**,
предоставляющая API к различным LLM.

Цель эксперимента — сравнить предсказания моделей с эталонной разметкой
и оценить, какие модели дают наиболее близкие результаты.

## Структура проекта и этапы работы

1. **Настройка API и моделей**  
    Подключение к OpenRouter API и определение набора тестируемых моделей.

2. **Описание шкалы и данных**  
    Формализация шкалы интроверсии–экстраверсии и требований к формату ответа моделей.

3. **Few-shot prompt engineering**  
    Формирование единого промпта на основе размеченных примеров из корпуса rusidiolect.

4. **Проведение экспериментов**  
   Применение одного набора тестовых текстов к каждой модели и извлечение численных 
   предсказаний и пояснений от моделей. 
   
5. **Анализ и сравнение моделей**  
    Расчёт абсолютных ошибок, агрегация результатов и ранжирование моделей.

6. **Сохранение результатов**  
    Экспорт результатов в csv.

7. **Выводы**  
    Обобщение полученных результатов и обсуждение ограничений.


### Импорты

In [72]:
import os
import time
import json
import re
from typing import List, Dict

from dotenv import load_dotenv
import requests
import pandas as pd
from openai import OpenAI

### 1. Настройка API и моделей

In [74]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY is not None, "OPENROUTER_API_KEY не найден в .env"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODELS = {
    # paid ones 
    "DeepSeek Chat": "deepseek/deepseek-v3.2",
    "Qwen3 Max": "qwen/qwen3-max",
    "Gemma 3 27B": "google/gemma-3-27b-it",
    "Llama 4 Scout": "meta-llama/llama-4-scout",
    "GPT-5.1": "openai/gpt-5.1",
    "Grok 4": "x-ai/grok-4",
    "Gemini 2.5 Pro": "google/gemini-2.5-pro",
    "Claude Sonnet 4.5": "anthropic/claude-sonnet-4.5",
    # free ones 
    # "LFM2.5-1.2B-Thinking": "liquid/lfm-2.5-1.2b-thinking:free",
    # "Molmo2 8B": "allenai/molmo-2-8b:free",
    # "MiMo-V2-Flash": "xiaomi/mimo-v2-flash:free",
    # "Nemotron 3 Nano 30B A3B": "nvidia/nemotron-3-nano-30b-a3b:free",
    # "Devstral 2 2512": "mistralai/devstral-2512:free",
    # "Trinity Mini": "arcee-ai/trinity-mini:free",
    # "Qwen3 Next 80B A3B Instruct": "qwen/qwen3-next-80b-a3b-instruct:free",
    # "gpt-oss-120b": "openai/gpt-oss-120b:free",
    # "Gemma 3 27B": "google/gemma-3-27b-it:free",
    # "DeepSeek: R1 0528": "deepseek/deepseek-r1-0528:free",
    # "Llama 3.3 70B Instruct": "meta-llama/llama-3.3-70b-instruct:free",
}

### 2. Описание шкалы и данных

#### Шкала интроверсии–экстраверсии

- **0–30** — выраженный интроверт  
- **31–45** — умеренный интроверт  
- **46–55** — средний уровень  
- **56–70** — умеренный экстраверт  
- **71–100** — выраженный экстраверт  

Модели должны вернуть:
1. Число от 0 до 100 
2. Краткое пояснение 

### 3. Few-shot prompt engineering

Для эксперимента используется единый few-shot набор для всех моделей.
Примеры взяты из корпуса rusidiolect и охватывают разные зоны шкалы интроверсии–экстраверсии, а также различные жанры текстов: описания картин и личные письма. 

In [75]:
SYSTEM_PROMPT = """
Ты — лингвист и психолог. Твоя задача — определить уровень интроверсии/экстраверсии автора 
текста по шкале от 0 до 100.
Интерпретация шкалы:
0–30 — выраженный интроверт
31–45 — умеренный интроверт
46–55 — средний
56–70 — умеренный экстраверт
71–100 — выраженный экстраверт

Ответь строго в следующем формате:
[число от 0 до 100]
[краткое пояснение]
"""


In [76]:
FEW_SHOT_EXAMPLES = [
    {
        "text": "Начнем сначала. На картине изображено 2 лица: молодое и старое. Теперь активируем режим поиска глубинного смысла и подумаем что же все это может означать (или даже символизировать). Возможно (если молодое лицо мужского пола, хотя однозначной идентификации гендера достичь возможным не представляется). На картине изображены мать и сын (или дочь, если мое предыдущее предположение все же является неверным). В таком случае на картине явно запечатлена тема «отцов и детей» (матерей в данном случае). Скорее всего мать наставляет сына (или дочь). Мысли и желания вполне очевидны. Беспокойство матери и беспечность (судя по отведенному взгляду) сына. Потом произойдет расставание. Есть и другой вариант, вырисовывающийся из положения лиц на картине. Возможно это гадалка, нашептывающая предсказания. Или старая советница какого-либо молодого короля. Может быть, она даже пытается им манипулировать. Мысли и последствия при подобном сценарии могут быть различны. Мысли советчицы могут варьироваться от государственного блага до своего собственного, а мысли короля и того шире. На картине не обязан действовать какой-либо сюжет. Это может быть процесс старения, воспоминания о молодости, сожаления об упущенных возможностях. Так что данное изображение может иметь самый широкий спектр толкований, вплоть до изображения доброго молодца и Бабы-Яги. Большое спасибо за внимание.",
        "score": 35
    },
    {
        "text": "Привет,  Данил! Как дела? Я надеюсь, что все хорошо. Последний месяц был очень напряженным и интересным. Я получил права на вождение автомобиля. Я целых полгода ходил на занятия. Это было утомительно, но это того стоило. Теперь я могу управлять авто. Это помогает делать много дел в один день. Побывать в сотнях новых мест, узнать много новых людей. Я понял смысл поговорки: «автомобиль не роскошь, а средство передвижения». И это правда! Учеба дается мне легко. Наша группа очень веселая и сильная. Мы сдаем завтра зачет по информатике. Через месяц у меня сессия. Немного волнуюсь, но да ладно! Расскажи о себе, мне все интересно. Жду ответа! Я знаю, что ты не любишь писать, но надеюсь на ответ. Может, позвонишь, мне будет приятно услышать твой голос. Мы давно не разговаривали по телефону. Твой номер не изменился? Или ты пользуешься только мобильным? Не пропадай. Пока!",
        "score": 39
    },
    {
        "text": "На мой взгляд, на картине изображены мать с сыном. Сын как будто бы принял для себя важное мнение. Возможно, он решил уехать из отчего дома. Взгляд его задумчив и направлен вперед, в будущее, в те дела и цели, которые ему предстоит осуществить. Мать же со своей стороны не поддерживает опрометчивое решение сына, но понимает, что не сможет его переубедить, поэтому ее взгляд задумчив и печален. Автор изобразил героев, смотрящих в разные стороны, будто хотел сказать этим, что судьбы их расходятся. Мать печальна, она понимает, что возможно она больше никогда не увидит сына. Она переживает за его судьбу. Сын же, напротив, полон энергии, новых идей, новых планов. Ему не терпится скорее вырваться из той жизни, которая его окружает, он хочет скорее окунуться в другую жизнь, познакомиться с новыми людьми, у него большие планы. Он рад предоставившемуся шансу. Ему совсем не хочется думать о чувствах матери. Конечно, ему жаль ее, но отступать от своей цели он не намерен. Сын вырос, ему стал слишком мал отчий дом. Он хочет свободы от материнской опеки. Мне кажется, он все же уедет, оставив мать совсем одну. А мать долго будет переживать утрату сына.",
        "score": 58
    },
    {
        "text": "Здравствуй мой милый друг. Как у тебя дела? Что нового? Я скучаю по тебе. У нас все тихо и спокойно. Недавно ходила на концерт. Получила много позитивных ощущений. Актер был просто супер: читал стихи, пел песни и танцевал. Очень понравилась вся труппа, живая музыка цепляла за душу, а танцы были просто класс, это неописуемо красиво. Расскажи как вы планируете провести новогодние каникулы? Может удастся собраться нашей веселой компанией. Я предлагаю всем встретиться на несколько дней в загородном доме. Пообщаемся как в старые добрые времена. А погода у нас замечательная. Снега намело по колено. Дети лепят снеговиков и строят снежные замки. Помнишь как на прошлый новый год мы наряжали снежную бабу в шапку твою и шарф а вместо носа была сосулька одетая в носок. Дети радовались. Кстати мы в этом году решили посадить елку во дворе, чтоб наряжать прям на улице. Детвора будет счастлива. До свидания. До встречи. Жду ответа. Твой друг.",
        "score": 60
    },
    {
        "text": "Мне представляется, что на картине изображены представители двух разных поколений. Умудренная опытом женщина и молодой человек (ее сын или внук). Судя по выражению лица женщины она вкрадчиво и мягко пытается донести до мужчины свои мысли, скорее хочет научить, предостеречь от ошибок, подсказать как лучше поступить. Ее взгляд ласковый  и теплый, полный любви и самых приятных чувств. В то же время в глазах угадываются нотки хитрости, качества, присущего многим женщинам. Молодой человек, по-моему с особым вниманием вслушивается в слова женщины. Он сосредоточен и задумчив. По моему мнению, данная картина демонстрирует преемственность поколений, передачу старшим поколением опыта жизни младшему поколению. Изображение демонстрирует не только преемственность поколений, но и уважение младшими старших. В наше время, когда люди становятся жестокими и черствыми, картина вызывает массу трепетных и теплых чувств... Заставляет вспомнить свою бабушку, ее ласковый тон и взгляд. Ее мудрые советы часто помогают мне сделать правильный выбор, не совершить ошибку. И даже сейчас, когда ее нет, порой вспоминаются наши душевные разговоры. Думаю, что и молодой человек, прислушается к словам женщины и сделает свой правильный выбор. Хочется отметить, насколько интересно лицо у женщины. Старое, морщинистое, но веселый, задорный взгляд с искринкой говорит о ясности ума женщины, умении позитивно относиться к жизни. Губы расплылись в улыбке. Лицо ласковое, лицо любящего человека. Такой человек может нести только добро.",
        "score": 46
    },
    {
        "text": "Здравствуй, дорогой мой друг! Я очень по тебе соскучилась! Сто лет тебя не видела! У меня произошло много, что интересного☺. Во-первых я поняла, что хочу выйти замуж за одного человека и обрести семью☺. Во-вторых, недавно познакомились с девчатами с одним парнем (мне кажется я ему понравилась), он очень добрый, заботливый и так далее, но он всего лишь в будущем только друг. Думаю, что в качестве друга он будет идеально смотреться (ахахаха), да и он будущий врач (неплохо таких знакомых иметь). В-третьих, наконец-таки принялась за вторую главу по диплому☺. Но над ним надо работать и работать. Скоро, кстати, сессия, но я надеюсь, что у меня по всем предметам будут автоматы. В-четвертых, скоро у нашего Ванюшки День Рождения (ему годик будет), поэтому помчу в Липецк (Вика отмечать хочет). Какие у тебя дела?? Что нового произошло?? Буду ждать от тебя ответа! Звони, пиши в социальных сетях, не забывай☺. Крепко обнимаю и целую!!!",
        "score": 64
    },
    {
        "text": "На картине я вижу молодого человека, который расположен на первом плане. На втором плане я вижу бабушку, которая подпирает голову рукой. Молодой человек не смотрит прямо, а смотрит вбок, видимо, что-то его заинтересовало. Молодой человек выглядит вполне приличным. На заднем плане изображена женщина пожилого возраста. Мне кажется, что она является добрым человеком. Я считаю, что скорее всего это бабушка и ее внук.",
        "score": 52
    },
]

### Формат ответа модели

Модели получают явные инструкции анализировать текст и формировать структурированный ответ.

In [77]:
def build_messages(text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for ex in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": ex["text"]})
        messages.append({"role": "assistant", "content": str(ex["score"])})

    messages.append({"role": "user", "content": text})
    return messages


In [78]:
# Используются 10 произвольных текстов с известным эталонным значением

texts_df = pd.DataFrame([
    {"id": 1, "text": "На переднем плане картины – молодой человек. Он смотрит в сторону с интересом, возможно, с какой-то долей беспокойства. Кажется, как будто сейчас он встанет и покинет пределы картины. Сразу за ним, почти вплотную, сидит женщина. Ее старческое лицо как будто усмехается, глаза смотрят прямо на меня. Рука поднесена ко рту, как будто она не хочет выболтать свои, (а может быть не свои) секреты. Она спокойна, ей некуда спешить. Старуха говорит: «Пусть парень уходит, ему не сидится на месте. А ты останься, мы поболтаем с тобой по душам». Почему-то мне не хочется ей верить, пожалуй, я бы пошла в направлении взгляда молодого человека.", "gold_score": 61},
    {"id": 2, "text": "Привет! Как дела? У меня все замечательно. Я съехала от мамы, живу в общаге теперь. Деньгами помогает папа и мама. Сначала они были против, но потом решили, что я уже повзрослела. Ты там как? Когда к нам обратно? В универе все хорошо. Скоро студенческая весна, но я участвовать не буду. Мне лень. Что у тебя нового? Как Кита? Наконец сестру уломали уйти с 9 класса. Она пойдет в техникум от моего Вуза на повара! Со старыми друзьями я не гуляю, они меня достали. Старшая скоро выйдет замуж. Вот готовимся к свадьбе. Купили ей платье очень красивое. Отмечать будем в усадьбе, типа выездной регистрации. А еще я собралась лететь в Париж! Давай со мной! А потом можно сгонять в Китай или Японию! Так, что жду, скучаю! Будем решать куда поедем☺", "gold_score": 56},
    {"id": 3, "text": "На картине изображены мать и сын. По ее взгляду видно, что она о чем-то сожалеет, а он не прислушивается к ее советам. Сын, видно, желает избежать этой ситуации. Он чувствует свою вину.", "gold_score": 47},
    {"id": 4, "text": "Привет! Этот месяц был очень интенсивным и веселым для меня. 23 февраля я виделась со своим парнем – Мишей, и подарила ему черную атласную бабочку. Он очень обрадовался этому подарку. Но веселей всего, что за этой бабочкой я ездила с родителями. Мама нормально отреагировала, потому  что я ей рассказывала про Мишу. А вот папа очень разозлился и сказал, что я только поступила в универ, а уже гуляю с кем-то и назвал его булочником. Но это ничего особо и не поменяло. На 8 Марта Миша подарил мне коробку конфет и тюльпаны. Мы провели хороший вечер вместе. 9 марта мы отмечали с девчонками 8 марта. Сначала я встретила на вокзале Марка и поехали в общагу за Попугайчиком. Наше путешествие продолжилось дорогой в магазин за тортиком и вином, а потом дорогой ко мне. Приехали ко мне и началось веселье!!! Музыка на полную и сидим распиваем вино с тортом. Я облилась мартини и пошла застирывать джинсы. Потом смотрели фильм «Зажги этим летом». Я уснула, а в это время звонит Миша, а я даже не услышала, и меня разбудил Попугайчик. Мы заснули. А с утра на немецкий и еще три пары. В эти выходные я поехала домой, потому что я приболела. На прошлой неделе было солнечное затмение, и в этот день я потеряла свой шарфик. В эту пятницу мы собрались с Юлей и сестрой погулять, зашли в «Эмиталь» и «Максимир». Я взяла задание по философии и поэтому купила книгу Льюиса «Хроники Нарнии». Зашли в точку зрения и я купила линзы для игры в футбол. В субботу я играла в футбол и гуляла с Мишей. В воскресенье я была на игре Миши.", "gold_score": 49},
    {"id": 5, "text": "Привет, Аня, мы давно с тобой не виделись, а общаемся мы только по интернету. Я поскорее хочу тебя увидеть: погулять с тобой, посмеяться. У меня все хорошо, в последнее время  дни у меня загружены, учеба. Но в целом хорошо. Особенно радует меня приближение весны, с каждым днем все теплее и теплее. Хочу, чтобы наступила настоящая весна, май и можно было погулять, походить по магазинам. Недавно я ходила в кино, на интересный фильм, мне очень понравился. Скоро пойду по магазинам, хочу купить новую одежду и обувь к лету. Приезжай ко мне, сходим вместе! Мои родные тоже чувствуют себя, слава Богу, хорошо. Я их навещаю каждые выходные, так как скучаю по дому, по друзьям. На этих выходных я тоже поеду домой. Когда еду домой я им постоянно привожу небольшие гостинцы. А как дела у тебя? Расскажи! Как учеба? Ты мне рассказывала, что вся в учебе, загруженные дни, подготовка к экзаменам.  Я знаю, что ты молодец, учишь уроки, и сдаешь экзамены на хорошие оценки. Так держать! Расскажи еще, про своих родных, как здоровье у них, работа, надеюсь, что все хорошо. Мы обе должны стараться в учебе, чтобы летом увидеться, побыть на даче, хорошо отдохнуть. У нас с тобой очень много смешных моментов, а их будет еще больше. Этим летом мы тоже, если все будет хорошо, будем ходить на речку, гулять, смеяться. И ты приедешь ко мне в гости. И я к тебе! Я тебя очень сильно люблю и скучаю! Пиши мне! До скорой встречи!", "gold_score": 57},
    {"id": 6, "text": "Привет, Машулька! Очень по тебе скучаю и хочу увидеть! Но ты далеко и увидимся мы только летом. Как у тебя дела? Как учеба? Как дома? И как у тебя душевное самочувствие? У меня все хорошо. На улице весна? Светит солнышко и все больше и больше дарит нам свое тепло. Уже скоро начнут появляться на деревьях почки, пробиваться трава. Я так хочу увидеть все зеленым и цветущим. Но учеба не дает этой красотой наслаждаться, получается только по выходным. Каждые выходные, которые я провожу дома, очень хорошие. И не хочу уезжать в этот шумный и полный суеты город. И ты каждый день должен соглашаться с его правилами. Со многими из которых ты не согласен. Каждый день тебя окружает толпа людей, занятых своими проблемами, серые дома, которые угнетают своим видом. Мне нравится учеба в университете. Конечно, что-то  нет, что-то получается, что-то нет. Но это нормально. Интересного почти ничего не происходит. Жду от тебя ответа. Люблю. Скучаю.", "gold_score": 47},
    {"id": 7, "text": "На картине изображены молодой человек и его мама. У них задумчивый вид. Возможно, произошел какой-либо конфликт или случилось что-то очень серьезное. У мамы доброжелательный взгляд. Она желает только добра своему сыну. Молодой человек отвел взгляд в сторону. Он чувствует свою вину; прислушивается к мнению матери. Мать и сын задумались, они хотят решить проблему, которая у них возникла. В конечном счете мать поговорит с сыном и они решат проблему, которая у них возникла.", "gold_score": 35},
    {"id": 8, "text": "Привет, Катя! Мы так давно с тобой не виделись, не разговаривали. Как жалко, что Воронеж нас разлучил! У меня столько накопилось информации, хочу с тобой всем поделиться. Начну с самого начала. Живу я сейчас со своим братом и его девушкой на съемной квартире. Квартира однокомнатная со старой мебелью, но уютная. Особенно мне нравится балкон, он такой большой, что в теплую погоду можно выносить стулья, стол и пить чай. Приезжай обязательно ко мне в гости и с тобой вместе попьем чай. Квартиру нам оплачивают мои братья, если бы не они, жили бы мы с Сережей по общежитиям, так как мама не в состоянии оплачивать всю сумму за квартиру, ну ты и сама прекрасно понимаешь зарплата сельского учителя не такая большая. В последнее время очень стала переживать за маму. Не знаю, как одна она там живет, скучно ей наверно. В выходные дни стараюсь как можно чаще приезжать к ней. Скучаю ужасно! Ну а в целом, Воронеж мне не очень нравится, или я просто к нему не привыкла. Все эти маршрутки, шум, пыль, люди, очень много людей – все это в новинку и непривычно. Хочу обратно в село. Мой брат говорит, что это у меня пройдет, нужно только привыкнуть. Что касается учебы в университете, то здесь у меня все хорошо. С первых дней познакомилась со всей группой, многие девчонки оказались очень простыми, доброжелательными. В основном все также не из Воронежа. Но хоть я и сдружилась с некоторыми, тебя мне все же не хватает. Жду твоего ответа! Напиши обязательно мне! Пока!", "gold_score": 32},
    {"id": 9, "text": "В недалеком прошлом многих девушек выдавали замуж поневоле. Их могли сватать при рождении. И они не могли перечить выбору их родителей. Девушки были совсем несчастны и не могли испытать настоящее чувство любви. Они всегда думали, что родители не желают им добра и счастья. На этой картине изображены мать и дочь. Она уже в зрелом возрасте и понимает, что поступки ее матери только во благо для нее. Ее жизнь размеренна и стабильна. Она не знает, что такое голод и нищета. Мать уберегла ее от всего этого и не дала почувствовать горечь жизни. Девушка все же стала злобной, неприступной женщиной, которая старалась держать все под своим контролем. А мать только и могла гордиться ею. Она добилась своего желанного, и теперь может спокойно дожить эту жизнь.", "gold_score": 55},
    {"id": 10, "text": "Привет моя любимая подружка, что-то давно я тебе не писала. У меня все хорошо, жизнь идет своим чередом. Последнее время загружена учебой и прочими домашними делами. Почти не гуляю. Только Коля меня и спасает, пытается вытащить погулять, хоть погода в последнее время не очень хорошая. Мы ходили в кино на вторую часть «Диверсанта». Мне очень понравилось, я была просто в восторге, ну ты же знаешь, как я люблю все эти фильмы, а вот Коля как всегда со своими шуточными вставками влезал, меня,  конечно,  это раздражало, но с ним фильм показался еще более ярким. Вот мы собираемся пойти на «Форсаж»,  там я думаю ему больше понравится, да и я уже жду его с нетерпением, (там же все мои любимые актеры собрались). Ой, а я себе уже накупила платьев и юбочек, вот жду тепла, а у тебя какие обновочки? Как ты вообще поживаешь? Как учеба? Хочу уже с тобой увидеться, погулять, поболтать. Дашулька тоже соскучилась ждет, когда ты приедешь и мы опять будем печь разные вкусности. Я, кстати, нашла много новых, интересных рецептов, вот жду не дождусь когда будем воплощать их в жизнь, а то с этой учебой суп приготовить некогда, не то что что-нибудь испечь. (сейчас начались контрольные так что задают много, времени мало, в общем,  печалька). Ну, ничего прорвемся! Как там твоя курсовая? И ты еще ходишь на волейбол? Как там поживает бабуля? Я так по всем соскучилась, хочется уже увидеться и поболтать. Вот а то в письме толком ничего не расскажешь, да я и не умею их писать, не люблю их ты же знаешь, что я считаю что надо видеть эмоции человека. Ну ладно, надо заканчивать. рассказывать нечего (ты лучше напиши, я знаю, что у тебя  всегда много событий, любишь ты приключения искать)☺. А лучше приезжай побыстрее!!! Скучаю, люблю, не дождусь когда ты приедешь☺. Твоя Кнопка☺.", "gold_score": 52},
])


In [79]:
def call_model_openrouter(model_id, messages, temperature=1.0):
    completion = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=temperature,
        extra_body={}
    )
    return completion.choices[0].message.content


In [80]:
def parse_response(text):
    if text is None:
        return None, None

    text = text.strip()
    match = re.match(r"^\s*(\d{1,3})\s*(.*)$", text, re.DOTALL)

    if not match:
        return None, text

    score = int(match.group(1))
    if not (0 <= score <= 100):
        return None, text

    explanation = match.group(2).strip()
    return score, explanation


### 4. Проведение экспериментов

Каждая модель:
- получает одинаковый системный промпт;
- анализирует один и тот же набор тестовых текстов;
- возвращает числовую оценку и краткое пояснение.

Для оценки качества используется абсолютная ошибка между предсказанием модели и эталонным значением из корпуса.

In [81]:
results = []

for text_id, row in texts_df.iterrows():
    text = row["text"]
    gold_score = row["gold_score"]

    for model_name, model_id in MODELS.items():
        messages = build_messages(text)

        try:
            response = call_model_openrouter(
                model_id=model_id,
                messages=messages,
                temperature=1.0
            )

            predicted_score, explanation = parse_response(response)
            abs_error = abs(predicted_score - gold_score) if predicted_score is not None else None

        except Exception as e:
            predicted_score = None
            explanation = f"API_ERROR: {str(e)}"
            abs_error = None

        results.append({
            "text_id": text_id,
            "model": model_name,
            "gold_score": gold_score,
            "predicted_score": predicted_score,
            "abs_error": abs_error,
            "explanation": explanation
        })


### 5. Анализ и сравнение моделей

Сравним модели по средней абсолютной ошибке.


In [82]:
df = pd.DataFrame(results)

model_ranking = (
    df.dropna(subset=["abs_error"])
      .groupby("model")["abs_error"]
      .mean()
      .reset_index()
      .sort_values("abs_error")
)

model_ranking


,model,abs_error
7,Qwen3 Max,11.1
1,DeepSeek Chat,12.0
2,GPT-5.1,12.9
6,Llama 4 Scout,13.9
4,Gemma 3 27B,14.0
3,Gemini 2.5 Pro,14.7
5,Grok 4,16.9
0,Claude Sonnet 4.5,17.5


In [83]:
df

,text_id,model,gold_score,predicted_score,abs_error,explanation
0,0,DeepSeek Chat,61,55,6,
1,0,Qwen3 Max,61,48,13,
2,0,Gemma 3 27B,61,48,13,
3,0,Llama 4 Scout,61,42,19,
4,0,GPT-5.1,61,48,13,Текст одновременно образный и рефлексивный: ав...
...,...,...,...,...,...,...
75,9,Llama 4 Scout,52,72,20,
76,9,GPT-5.1,52,63,11,"Эмоциональное, развернутое письмо с обилием во..."
77,9,Grok 4,52,68,16,"Текст полон энтузиазма по поводу встреч, общен..."
78,9,Gemini 2.5 Pro,52,68,16,Выраженная ориентация на социальные контакты и...


### 6. Сохранение результатов

In [84]:
df.to_csv("llm_introversion_results.csv", index=False)

### 7. Выводы

Эксперимент показал, что современные LLM способны в разумных пределах восстанавливать личностные характеристики автора текста на основе психолингвистических признаков.

При этом:
- некоторые модели склонны переоценивать экстраверсию, ориентируясь на эмоциональность текста;
- другие, напротив, занижают оценки, фокусируясь на интроспекции даже при наличии активных социальных действий;
- качество моделей существенно различается даже при одинаковых условиях;
- лучшая модель по средней ошибке не всегда даёт наиболее убедительные объяснения;
- задача оценки личностных черт остаётся сложной и чувствительной к интерпретации.

Ограничения исследования
- небольшой объём тестовых данных;
- зависимость результатов от конкретной формулировки промпта.

В целом проект демонстрирует потенциал LLM для задач психолингвистического анализа, а также подчёркивает необходимость сочетания количественных и качественных методов оценки.
